# Pandas Data Exploration Playground

**Level:** Beginner  
**Estimated Time:** 120-150 minutes  
**Category:** Python & Data Fundamentals  
**Tags:** Pandas, Data Cleaning, Exploration, Time Series

---

## Introduction

Welcome to the **Pandas Data Exploration Playground**! Pandas is the cornerstone of financial data analysis in Python, providing powerful tools to manipulate, analyze, and visualize time series data.

### Why Pandas for Finance?

In quantitative finance, you'll work with:
- **Time series data**: Stock prices, returns, volume
- **Cross-sectional data**: Portfolio holdings, fundamental metrics
- **Panel data**: Multi-asset data over time
- **Event data**: Trades, orders, corporate actions

Pandas excels at all of these, offering:
- **Fast vectorized operations**: 100-1000x faster than loops
- **Time series functionality**: Date indexing, resampling, rolling windows
- **Data alignment**: Automatic alignment by index/date
- **Missing data handling**: Built-in methods for NaN values
- **Integration**: Works seamlessly with NumPy, Matplotlib, scikit-learn

### Learning Objectives

By the end of this notebook, you will:

1. ✅ Create and manipulate DataFrames for financial data
2. ✅ Inspect, filter, and select data efficiently
3. ✅ Clean and handle missing data properly
4. ✅ Engineer features: returns, indicators, rolling statistics
5. ✅ Group, aggregate, and resample time series
6. ✅ Merge and join multiple datasets
7. ✅ Apply functions and vectorized operations
8. ✅ Understand performance optimization techniques

### Prerequisites

- Basic Python knowledge (variables, functions, loops)
- Understanding of financial concepts (prices, returns, volume)
- Familiarity with NumPy arrays (helpful but not required)

### Real-World Applications

**Portfolio Management:**
- Calculate portfolio returns and rebalancing
- Track holdings and cash flows over time
- Analyze factor exposures across assets

**Risk Management:**
- Compute rolling volatility and correlation
- Identify outliers and stress periods
- Calculate VaR and drawdowns

**Algorithmic Trading:**
- Process tick data and order books
- Calculate technical indicators (moving averages, RSI)
- Backtest trading strategies

**Research:**
- Clean and merge multiple data sources
- Analyze corporate actions and dividends
- Study cross-sectional factor performance

---

## Table of Contents

1. **Creating Financial DataFrames** - From dictionaries, arrays, time series
2. **Data Inspection** - Info, describe, head/tail
3. **Selection & Filtering** - Columns, rows, boolean indexing
4. **Data Cleaning** - Missing values, duplicates, outliers
5. **Feature Engineering** - Returns, indicators, rolling windows
6. **Grouping & Aggregation** - Resampling, GroupBy
7. **Merging Data** - Concat, merge, join
8. **Advanced Operations** - Apply, pivot, transform
9. **Performance Optimization** - Vectorization, memory
10. **Summary & Practice** - Key takeaways, exercises

Let's dive in!

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print(f"Pandas version: {pd.__version__}")
print("Setup complete!")

## 1. Creating Financial DataFrames

### From Dictionaries

In [ ]:
# Create sample stock data
stock_data = {
    'Ticker': ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'META'],
    'Price': [178.50, 140.25, 378.90, 145.30, 485.75],
    'Volume': [52000000, 28000000, 35000000, 48000000, 18000000],
    'Market_Cap_B': [2800, 1750, 2820, 1500, 1200],
    'Sector': ['Technology', 'Technology', 'Technology', 'Consumer', 'Technology'],
    'PE_Ratio': [29.5, 26.8, 35.2, 72.4, 28.9]
}

df_stocks = pd.DataFrame(stock_data)
print("Stock DataFrame:")
print(df_stocks)
print(f"\nShape: {df_stocks.shape}")
print(f"\nData Types:\n{df_stocks.dtypes}")

### Time Series Data

In [ ]:
# Generate time series of OHLCV data
np.random.seed(42)
dates = pd.date_range(start='2024-01-01', end='2024-12-31', freq='D')
n_days = len(dates)

# Simulate price with GBM
returns = np.random.normal(0.0005, 0.015, n_days)
price = 100 * np.exp(np.cumsum(returns))

# Generate OHLCV
ohlcv_data = pd.DataFrame({
    'Date': dates,
    'Open': price * np.random.uniform(0.995, 1.005, n_days),
    'High': price * np.random.uniform(1.000, 1.020, n_days),
    'Low': price * np.random.uniform(0.980, 1.000, n_days),
    'Close': price,
    'Volume': np.random.randint(10000000, 100000000, n_days)
})

# Set Date as index
ohlcv_data.set_index('Date', inplace=True)

print("OHLCV Time Series:")
print(ohlcv_data.head(10))
print(f"\nDate range: {ohlcv_data.index[0]} to {ohlcv_data.index[-1]}")
print(f"Total trading days: {len(ohlcv_data)}")

## 2. Data Inspection and Summary

### Basic Info

In [ ]:
print("=" * 60)
print("DATA INSPECTION")
print("=" * 60)

# Info
print("\nDataFrame Info:")
print(ohlcv_data.info())

# Describe
print("\nStatistical Summary:")
print(ohlcv_data.describe())

# Custom statistics
print("\nCustom Statistics:")
print(f"Mean Close Price: ${ohlcv_data['Close'].mean():.2f}")
print(f"Median Close Price: ${ohlcv_data['Close'].median():.2f}")
print(f"Std Dev: ${ohlcv_data['Close'].std():.2f}")
print(f"Min Price: ${ohlcv_data['Close'].min():.2f}")
print(f"Max Price: ${ohlcv_data['Close'].max():.2f}")
print(f"Price Range: ${ohlcv_data['Close'].max() - ohlcv_data['Close'].min():.2f}")

## 3. Data Selection and Filtering

### Selecting Columns and Rows

In [ ]:
# Select specific columns
prices_only = ohlcv_data[['Open', 'High', 'Low', 'Close']]
print("Price columns only:")
print(prices_only.head())

# Select rows by position
print("\nFirst 5 rows:")
print(ohlcv_data.head())

print("\nLast 5 rows:")
print(ohlcv_data.tail())

# Select rows by label (date)
print("\nJanuary 2024 data:")
jan_data = ohlcv_data.loc['2024-01-01':'2024-01-31']
print(jan_data.head())
print(f"January had {len(jan_data)} trading days")

### Boolean Filtering

In [ ]:
# Filter high volume days
high_volume = ohlcv_data[ohlcv_data['Volume'] > 60000000]
print(f"High volume days (>60M): {len(high_volume)}")
print(high_volume.head())

# Filter by price range
price_range = ohlcv_data[(ohlcv_data['Close'] > 100) & (ohlcv_data['Close'] < 110)]
print(f"\nDays with price between $100-$110: {len(price_range)}")

# Filter by date and condition
q1_high_vol = ohlcv_data.loc['2024-01':'2024-03'][ohlcv_data['Volume'] > 50000000]
print(f"\nQ1 high volume days: {len(q1_high_vol)}")

# Multiple conditions
volatile_days = ohlcv_data[
    ((ohlcv_data['High'] - ohlcv_data['Low']) / ohlcv_data['Close'] > 0.03) &
    (ohlcv_data['Volume'] > 50000000)
]
print(f"\nVolatile days (>3% range + high volume): {len(volatile_days)}")

## 4. Data Cleaning

### Handling Missing Data

In [ ]:
# Create a copy with missing values
df_missing = ohlcv_data.copy()

# Randomly insert NaN values
np.random.seed(42)
missing_indices = np.random.choice(df_missing.index, size=20, replace=False)
df_missing.loc[missing_indices, 'Close'] = np.nan
df_missing.loc[missing_indices[:10], 'Volume'] = np.nan

print("Missing value summary:")
print(df_missing.isnull().sum())
print(f"\nTotal missing values: {df_missing.isnull().sum().sum()}")

# Visualize missing data
plt.figure(figsize=(12, 4))
plt.imshow(df_missing.isnull().T, cmap='RdYlGn_r', aspect='auto', interpolation='none')
plt.yticks(range(len(df_missing.columns)), df_missing.columns)
plt.xlabel('Row Index')
plt.title('Missing Data Visualization (Red = Missing)')
plt.colorbar(label='Missing (1) vs Present (0)')
plt.tight_layout()
plt.show()

In [ ]:
# Different strategies for handling missing data

# 1. Drop rows with any NaN
df_dropped = df_missing.dropna()
print(f"After dropping NaN rows: {len(df_dropped)} rows (from {len(df_missing)})")

# 2. Forward fill (use last valid value)
df_ffill = df_missing.fillna(method='ffill')
print(f"After forward fill: {df_ffill.isnull().sum().sum()} missing values")

# 3. Backward fill
df_bfill = df_missing.fillna(method='bfill')
print(f"After backward fill: {df_bfill.isnull().sum().sum()} missing values")

# 4. Interpolate (linear)
df_interp = df_missing.interpolate(method='linear')
print(f"After interpolation: {df_interp.isnull().sum().sum()} missing values")

# 5. Fill with mean
df_mean = df_missing.fillna(df_missing.mean())
print(f"After mean fill: {df_mean.isnull().sum().sum()} missing values")

# Compare methods
sample_idx = missing_indices[0]
print(f"\nComparison for {sample_idx}:")
print(f"Original (NaN): {df_missing.loc[sample_idx, 'Close']}")
print(f"Forward fill: {df_ffill.loc[sample_idx, 'Close']:.2f}")
print(f"Interpolate: {df_interp.loc[sample_idx, 'Close']:.2f}")
print(f"Mean fill: {df_mean.loc[sample_idx, 'Close']:.2f}")

## 5. Feature Engineering

### Creating New Columns

In [ ]:
# Work with clean data
df = ohlcv_data.copy()

# Simple returns
df['Return'] = df['Close'].pct_change()

# Log returns
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

# Daily range
df['Daily_Range'] = df['High'] - df['Low']
df['Range_Pct'] = (df['Daily_Range'] / df['Close']) * 100

# Intraday gap
df['Gap'] = df['Open'] - df['Close'].shift(1)
df['Gap_Pct'] = (df['Gap'] / df['Close'].shift(1)) * 100

# Volume change
df['Volume_Change'] = df['Volume'].pct_change()

# Price change from open
df['Price_Change'] = df['Close'] - df['Open']
df['Price_Change_Pct'] = (df['Price_Change'] / df['Open']) * 100

print("DataFrame with engineered features:")
print(df[['Close', 'Return', 'Daily_Range', 'Range_Pct', 'Volume_Change']].head(10))

# Summary statistics
print("\nFeature Statistics:")
print(df[['Return', 'Range_Pct', 'Volume_Change']].describe())

### Rolling Window Calculations

In [ ]:
# Moving averages
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()
df['SMA_200'] = df['Close'].rolling(window=200).mean()

# Exponential moving average
df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()

# Rolling volatility (20-day)
df['Volatility_20'] = df['Return'].rolling(window=20).std() * np.sqrt(252)

# Rolling volume average
df['Avg_Volume_20'] = df['Volume'].rolling(window=20).mean()
df['Volume_Ratio'] = df['Volume'] / df['Avg_Volume_20']

# Bollinger Bands
df['BB_Middle'] = df['Close'].rolling(window=20).mean()
df['BB_Std'] = df['Close'].rolling(window=20).std()
df['BB_Upper'] = df['BB_Middle'] + (2 * df['BB_Std'])
df['BB_Lower'] = df['BB_Middle'] - (2 * df['BB_Std'])

# %B (position within Bollinger Bands)
df['Percent_B'] = (df['Close'] - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'])

print("Moving Averages and Indicators:")
print(df[['Close', 'SMA_20', 'SMA_50', 'EMA_20', 'Volatility_20']].tail(10))

## 6. Grouping and Aggregation

### Resampling Time Series

In [ ]:
# Resample to weekly data
weekly = df.resample('W').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum',
    'Return': 'sum'
})

print("Weekly OHLCV Data:")
print(weekly.head(10))

# Resample to monthly
monthly = df.resample('M').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'mean',
    'Return': lambda x: (1 + x).prod() - 1  # Compound return
})

print("\nMonthly OHLCV Data:")
print(monthly)

# Calculate monthly statistics
print("\nMonthly Return Statistics:")
print(f"Best month: {monthly['Return'].max():.2%}")
print(f"Worst month: {monthly['Return'].min():.2%}")
print(f"Average monthly return: {monthly['Return'].mean():.2%}")

### GroupBy Operations

In [ ]:
# Add categorical columns
df['Month'] = df.index.month
df['Quarter'] = df.index.quarter
df['Day_of_Week'] = df.index.dayofweek
df['Week_of_Year'] = df.index.isocalendar().week

# Group by month
monthly_stats = df.groupby('Month').agg({
    'Return': ['mean', 'std', 'min', 'max'],
    'Volume': 'mean',
    'Range_Pct': 'mean'
})

print("Monthly Statistics:")
print(monthly_stats)

# Group by day of week
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
dow_stats = df.groupby('Day_of_Week')['Return'].agg(['mean', 'std', 'count'])
dow_stats.index = [day_names[i] for i in dow_stats.index]

print("\nDay of Week Statistics:")
print(dow_stats)

# Visualize day of week effect
plt.figure(figsize=(12, 5))
plt.bar(dow_stats.index, dow_stats['mean'] * 100, color='skyblue', edgecolor='black')
plt.axhline(y=0, color='red', linestyle='--', linewidth=1)
plt.title('Average Return by Day of Week', fontsize=14, fontweight='bold')
plt.xlabel('Day of Week')
plt.ylabel('Average Return (%)')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 7. Merging and Joining Data

### Combining Multiple Assets

In [ ]:
# Create data for multiple assets
np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=100, freq='D')

# Asset 1: Tech stock
tech_returns = np.random.normal(0.001, 0.02, 100)
tech_prices = 150 * np.exp(np.cumsum(tech_returns))
df_tech = pd.DataFrame({
    'Date': dates,
    'TECH_Close': tech_prices,
    'TECH_Volume': np.random.randint(50000000, 100000000, 100)
}).set_index('Date')

# Asset 2: Financial stock
fin_returns = np.random.normal(0.0005, 0.015, 100)
fin_prices = 80 * np.exp(np.cumsum(fin_returns))
df_fin = pd.DataFrame({
    'Date': dates,
    'FIN_Close': fin_prices,
    'FIN_Volume': np.random.randint(30000000, 70000000, 100)
}).set_index('Date')

# Asset 3: Energy stock
energy_returns = np.random.normal(0.0003, 0.025, 100)
energy_prices = 60 * np.exp(np.cumsum(energy_returns))
df_energy = pd.DataFrame({
    'Date': dates,
    'ENERGY_Close': energy_prices,
    'ENERGY_Volume': np.random.randint(40000000, 80000000, 100)
}).set_index('Date')

# Merge all datasets
df_combined = pd.concat([df_tech, df_fin, df_energy], axis=1)

print("Combined Multi-Asset DataFrame:")
print(df_combined.head())
print(f"\nShape: {df_combined.shape}")
print(f"Columns: {list(df_combined.columns)}")

In [ ]:
# Calculate returns for all assets
df_combined['TECH_Return'] = df_combined['TECH_Close'].pct_change()
df_combined['FIN_Return'] = df_combined['FIN_Close'].pct_change()
df_combined['ENERGY_Return'] = df_combined['ENERGY_Close'].pct_change()

# Correlation matrix
returns_cols = ['TECH_Return', 'FIN_Return', 'ENERGY_Return']
correlation = df_combined[returns_cols].corr()

print("Return Correlation Matrix:")
print(correlation)

# Visualize correlations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
ax = axes[0]
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, 
            vmin=-1, vmax=1, square=True, ax=ax, fmt='.3f')
ax.set_title('Return Correlation Heatmap', fontsize=14, fontweight='bold')

# Scatter plot
ax = axes[1]
ax.scatter(df_combined['TECH_Return'], df_combined['FIN_Return'], 
          alpha=0.5, label='TECH vs FIN')
ax.set_xlabel('TECH Return')
ax.set_ylabel('FIN Return')
ax.set_title('Return Scatter Plot', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 8. Advanced DataFrame Operations

### Pivot Tables

In [ ]:
# Create transaction data
transactions = pd.DataFrame({
    'Date': pd.date_range('2024-01-01', periods=50, freq='D').repeat(3),
    'Ticker': ['AAPL', 'GOOGL', 'MSFT'] * 50,
    'Quantity': np.random.randint(-100, 100, 150),
    'Price': np.random.uniform(100, 200, 150)
})

transactions['Value'] = transactions['Quantity'] * transactions['Price']

print("Transaction Data:")
print(transactions.head(10))

# Pivot table: Total value by ticker and date
pivot = pd.pivot_table(
    transactions,
    values='Value',
    index=transactions['Date'].dt.to_period('W'),
    columns='Ticker',
    aggfunc='sum',
    fill_value=0
)

print("\nWeekly Transaction Value by Ticker:")
print(pivot.head(10))

### Apply and Lambda Functions

In [ ]:
# Custom function to categorize returns
def categorize_return(ret):
    if pd.isna(ret):
        return 'Unknown'
    elif ret > 0.02:
        return 'Strong Up'
    elif ret > 0:
        return 'Up'
    elif ret > -0.02:
        return 'Down'
    else:
        return 'Strong Down'

# Apply function
df['Return_Category'] = df['Return'].apply(categorize_return)

# Count categories
category_counts = df['Return_Category'].value_counts()
print("Return Categories:")
print(category_counts)

# Lambda function for quick calculations
df['Return_Squared'] = df['Return'].apply(lambda x: x**2 if not pd.isna(x) else 0)

# Apply to multiple columns
price_cols = ['Open', 'High', 'Low', 'Close']
df_normalized = df[price_cols].apply(lambda x: (x - x.mean()) / x.std())

print("\nNormalized Prices (Z-scores):")
print(df_normalized.head())

## 9. Performance Tips

### Vectorized Operations vs Loops

In [ ]:
import time

# Create test data
test_df = pd.DataFrame({
    'A': np.random.randn(10000),
    'B': np.random.randn(10000)
})

# Method 1: Loop (slow)
start = time.time()
result_loop = []
for i in range(len(test_df)):
    result_loop.append(test_df.iloc[i]['A'] * test_df.iloc[i]['B'])
test_df['C_loop'] = result_loop
loop_time = time.time() - start

# Method 2: Vectorized (fast)
start = time.time()
test_df['C_vectorized'] = test_df['A'] * test_df['B']
vectorized_time = time.time() - start

print(f"Loop time: {loop_time:.4f} seconds")
print(f"Vectorized time: {vectorized_time:.4f} seconds")
print(f"Speedup: {loop_time / vectorized_time:.1f}x")

# Verify results are the same
print(f"\nResults match: {np.allclose(test_df['C_loop'], test_df['C_vectorized'])}")

## 10. Comprehensive Visualizations

Let's create publication-quality visualizations of our financial data analysis.

## Summary and Key Takeaways

Congratulations! You've mastered Pandas for financial data analysis. Let's review what you learned.

### Key Concepts Covered

#### 1. DataFrame Creation & Structure
- **From dictionaries**: Quick setup for small datasets
- **Time series indexing**: DatetimeIndex for financial data
- **OHLCV data**: Standard structure for market data
- **Multi-asset DataFrames**: Panel data organization

#### 2. Data Inspection & Selection
- **`.info()` and `.describe()`**: Quick data overview
- **`.loc` and `.iloc`**: Label and position-based indexing
- **Boolean filtering**: Complex conditional selection
- **Date slicing**: Time-based data retrieval

#### 3. Data Cleaning
- **Missing values**: `.fillna()`, `.interpolate()`, `.dropna()`
- **Forward/backward fill**: Time series specific methods
- **Outlier detection**: Statistical methods
- **Data validation**: Type checking and constraints

#### 4. Feature Engineering
- **Returns calculation**: `.pct_change()` and log returns
- **Rolling windows**: `.rolling()` for moving statistics
- **Technical indicators**: SMA, EMA, Bollinger Bands
- **Date features**: Extract month, day of week, quarter

#### 5. Grouping & Aggregation
- **`.groupby()`**: Split-apply-combine pattern
- **`.resample()`**: Time-based aggregation (daily → weekly → monthly)
- **Multiple aggregations**: Different functions per column
- **Custom aggregation functions**: Lambda and named functions

#### 6. Merging & Joining
- **`.concat()`**: Stack DataFrames vertically or horizontally
- **`.merge()`**: SQL-like joins (inner, outer, left, right)
- **Index alignment**: Automatic date alignment
- **Multi-asset analysis**: Correlation and covariance

#### 7. Advanced Operations
- **`.apply()`**: Row/column-wise custom functions
- **`.pivot_table()`**: Reshape data for analysis
- **Vectorization**: 100-1000x faster than loops
- **Memory optimization**: Data types and chunking

### Critical Pandas Patterns for Finance

```python
# 1. Calculate returns properly
df['return'] = df['Close'].pct_change()  # Simple return
df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))  # Log return

# 2. Handle missing data in time series
df_filled = df.fillna(method='ffill')  # Forward fill (most common)

# 3. Compute rolling statistics
df['vol_20d'] = df['return'].rolling(20).std() * np.sqrt(252)

# 4. Resample to different frequencies
monthly = df.resample('M').agg({'Close': 'last', 'Volume': 'sum'})

# 5. Group by categorical variables
sector_stats = df.groupby('Sector')['Return'].agg(['mean', 'std', 'count'])

# 6. Merge multiple data sources
combined = pd.merge(prices, fundamentals, on=['Date', 'Ticker'], how='inner')

# 7. Vectorized operations (avoid loops!)
df['portfolio_value'] = (df[asset_cols] * weights).sum(axis=1)
```

### Common Pitfalls to Avoid

1. **Using loops instead of vectorization**: Always prefer vectorized operations
2. **Ignoring the index**: DatetimeIndex is crucial for time series operations
3. **Chained assignment**: Use `.loc` to avoid `SettingWithCopyWarning`
4. **Not handling missing data**: Always check for and handle NaN values
5. **Inefficient resampling**: Use built-in `.resample()` instead of manual grouping
6. **Memory issues**: Use appropriate data types (float32 vs float64, category dtype)
7. **Not validating merges**: Always check merge results for duplicates and missing values

### Performance Best Practices

```python
# ❌ SLOW: Loop over rows
for i in range(len(df)):
    df.loc[i, 'result'] = df.loc[i, 'A'] * df.loc[i, 'B']

# ✅ FAST: Vectorized operation
df['result'] = df['A'] * df['B']

# ❌ SLOW: Appending in loop
for row in data:
    df = df.append(row)

# ✅ FAST: Create list then DataFrame
df = pd.DataFrame(data_list)

# Memory optimization
df['Category'] = df['Category'].astype('category')  # For repeated strings
df['Price'] = df['Price'].astype('float32')  # If precision allows
```

### Real-World Applications

You can now:

1. **Load and clean** market data from multiple sources
2. **Calculate** returns, volatility, and risk metrics
3. **Build** technical indicators and trading signals
4. **Analyze** portfolio performance and attribution
5. **Resample** data to different frequencies (tick → minute → daily)
6. **Merge** fundamental data with price data
7. **Group** and analyze by sectors, industries, or factors
8. **Optimize** code for production-level performance

### Next Steps

**Continue Learning:**
1. **Advanced Time Series**: ARIMA, GARCH, state space models
2. **Portfolio Optimization**: Mean-variance, risk parity, Black-Litterman
3. **Machine Learning**: scikit-learn integration with Pandas
4. **Data APIs**: yfinance, Alpha Vantage, Bloomberg
5. **Big Data**: Dask for datasets larger than memory
6. **Backtesting**: Zipline, Backtrader frameworks

**Practice Exercises:**

See the exercises section below for hands-on practice problems!

---

## Practice Exercises

### Exercise 1: Returns Analysis
Load a dataset and calculate:
- Simple and log returns
- Cumulative returns
- Annualized return and volatility
- Sharpe ratio (assuming 2% risk-free rate)

### Exercise 2: Moving Averages
Create a trading signal based on:
- 20-day and 50-day moving averages
- Generate buy signals when 20-day > 50-day
- Calculate strategy returns vs buy-and-hold

### Exercise 3: Portfolio Rebalancing
Build a portfolio with monthly rebalancing:
- Start with equal weights across 5 assets
- Rebalance to equal weights at month-end
- Compare performance vs buy-and-hold

### Exercise 4: Risk Analysis
For a given dataset, calculate:
- Rolling 30-day volatility
- Maximum drawdown and drawdown duration
- 95% and 99% VaR
- Identify worst 10 days

### Exercise 5: Multi-Asset Analysis
Merge 3 different asset datasets and:
- Calculate correlation matrix
- Find best and worst diversification pairs
- Compute efficient frontier (2-asset case)
- Visualize with scatter plot

---

## References and Further Reading

### Books
1. **McKinney, W. (2022).** *Python for Data Analysis* (3rd ed.). O'Reilly Media.
   - Authoritative guide by Pandas creator
   - Covers all DataFrame operations in depth
   
2. **Hilpisch, Y. (2018).** *Python for Finance* (2nd ed.). O'Reilly Media.
   - Financial applications of Pandas
   - Real-world examples and case studies

3. **VanderPlas, J. (2016).** *Python Data Science Handbook*. O'Reilly Media.
   - Chapter on Pandas fundamentals
   - Integration with NumPy and Matplotlib

### Online Resources
- [Pandas Official Documentation](https://pandas.pydata.org/docs/) - Comprehensive reference
- [10 Minutes to Pandas](https://pandas.pydata.org/docs/user_guide/10min.html) - Quick start guide
- [Pandas Cheat Sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) - Reference card
- [Modern Pandas](https://tomaugspurger.github.io/modern-1-intro.html) - Best practices series

### Video Tutorials
- [Corey Schafer's Pandas Tutorials](https://www.youtube.com/playlist?list=PL-osiE80TeTsWmV9i9c58mdDCSskIFdDS)
- [Data School Pandas Videos](https://www.youtube.com/user/dataschool)

---

**Congratulations on completing the Pandas Data Exploration Playground!** 

You now have the skills to handle any financial dataset with confidence. Keep practicing, and remember: **vectorization is your friend!**

In [ ]:
# Complete portfolio analysis workflow
print("=" * 70)
print("COMPREHENSIVE PORTFOLIO ANALYSIS WORKFLOW")
print("=" * 70)

# Step 1: Create multi-asset portfolio
tickers = ['TECH', 'FIN', 'ENERGY', 'HEALTH', 'CONSUMER']
n_assets = len(tickers)
n_days = 252  # 1 trading year

np.random.seed(42)
dates_portfolio = pd.date_range('2024-01-01', periods=n_days, freq='B')  # Business days

# Simulate correlated returns
correlation_matrix = np.array([
    [1.00, 0.65, 0.45, 0.55, 0.60],  # TECH
    [0.65, 1.00, 0.50, 0.48, 0.52],  # FIN
    [0.45, 0.50, 1.00, 0.40, 0.35],  # ENERGY
    [0.55, 0.48, 0.40, 1.00, 0.58],  # HEALTH
    [0.60, 0.52, 0.35, 0.58, 1.00]   # CONSUMER
])

# Cholesky decomposition for correlated returns
L = np.linalg.cholesky(correlation_matrix)
uncorrelated = np.random.normal(0, 0.015, (n_days, n_assets))
correlated_returns = uncorrelated @ L.T

# Add different drift rates
drift = np.array([0.0008, 0.0005, 0.0006, 0.0007, 0.0004])
returns_matrix = correlated_returns + drift

# Create DataFrame
df_portfolio = pd.DataFrame(
    returns_matrix,
    index=dates_portfolio,
    columns=tickers
)

# Generate prices from returns (starting at $100)
df_prices_port = pd.DataFrame(index=dates_portfolio, columns=tickers)
for ticker in tickers:
    df_prices_port[ticker] = 100 * np.exp(df_portfolio[ticker].cumsum())

print("\n1. PORTFOLIO COMPOSITION")
print("-" * 70)
print(f"Assets: {', '.join(tickers)}")
print(f"Period: {dates_portfolio[0].date()} to {dates_portfolio[-1].date()}")
print(f"Trading days: {n_days}")

# Step 2: Define portfolio weights
weights = np.array([0.25, 0.20, 0.15, 0.25, 0.15])  # Sum to 1.0
initial_capital = 1_000_000

print("\n2. ALLOCATION")
print("-" * 70)
for ticker, weight in zip(tickers, weights):
    allocation = initial_capital * weight
    print(f"{ticker:10s}: {weight:5.1%} (${allocation:>12,.0f})")
print(f"{'TOTAL':10s}: {weights.sum():5.1%} (${initial_capital:>12,.0f})")

# Step 3: Calculate portfolio returns
df_portfolio_returns = (df_portfolio * weights).sum(axis=1)
df_portfolio_value = initial_capital * (1 + df_portfolio_returns).cumprod()

# Step 4: Performance metrics
total_return = (df_portfolio_value.iloc[-1] / initial_capital) - 1
annualized_return = (1 + total_return) ** (252 / n_days) - 1
volatility = df_portfolio_returns.std() * np.sqrt(252)
sharpe_ratio = (annualized_return - 0.02) / volatility  # Assuming 2% risk-free rate

# Maximum drawdown
cumulative = (1 + df_portfolio_returns).cumprod()
running_max = cumulative.expanding().max()
drawdown = (cumulative - running_max) / running_max
max_drawdown = drawdown.min()

# Downside deviation
downside_returns = df_portfolio_returns[df_portfolio_returns < 0]
downside_deviation = downside_returns.std() * np.sqrt(252)
sortino_ratio = (annualized_return - 0.02) / downside_deviation if downside_deviation > 0 else np.nan

# Value at Risk (95% confidence)
var_95 = np.percentile(df_portfolio_returns, 5)
cvar_95 = df_portfolio_returns[df_portfolio_returns <= var_95].mean()

print("\n3. PERFORMANCE METRICS")
print("-" * 70)
print(f"Total Return:          {total_return:>10.2%}")
print(f"Annualized Return:     {annualized_return:>10.2%}")
print(f"Volatility (Ann.):     {volatility:>10.2%}")
print(f"Sharpe Ratio:          {sharpe_ratio:>10.2f}")
print(f"Sortino Ratio:         {sortino_ratio:>10.2f}")
print(f"Maximum Drawdown:      {max_drawdown:>10.2%}")
print(f"VaR (95%):             {var_95:>10.2%}")
print(f"CVaR (95%):            {cvar_95:>10.2%}")
print(f"Best Day:              {df_portfolio_returns.max():>10.2%}")
print(f"Worst Day:             {df_portfolio_returns.min():>10.2%}")
print(f"Positive Days:         {(df_portfolio_returns > 0).sum():>10d} ({(df_portfolio_returns > 0).mean():.1%})")

# Step 5: Asset-level statistics
print("\n4. INDIVIDUAL ASSET PERFORMANCE")
print("-" * 70)
print(f"{'Asset':<10} {'Return':>10} {'Volatility':>12} {'Sharpe':>10} {'Weight':>10}")
print("-" * 70)

for ticker, weight in zip(tickers, weights):
    asset_returns = df_portfolio[ticker]
    ann_ret = asset_returns.mean() * 252
    ann_vol = asset_returns.std() * np.sqrt(252)
    asset_sharpe = (ann_ret - 0.02) / ann_vol
    print(f"{ticker:<10} {ann_ret:>9.2%} {ann_vol:>11.2%} {asset_sharpe:>10.2f} {weight:>9.1%}")

# Step 6: Correlation analysis
print("\n5. CORRELATION MATRIX")
print("-" * 70)
actual_corr = df_portfolio.corr()
print(actual_corr.round(3))

# Step 7: Risk contribution
# Marginal contribution to risk
portfolio_variance = (df_portfolio_returns).var()
marginal_contrib = []
for i, ticker in enumerate(tickers):
    # Covariance of asset with portfolio
    cov_with_port = df_portfolio[ticker].cov(df_portfolio_returns)
    marginal_contrib.append(weights[i] * cov_with_port / portfolio_variance)

marginal_contrib = np.array(marginal_contrib)

print("\n6. RISK CONTRIBUTION")
print("-" * 70)
print(f"{'Asset':<10} {'Weight':>10} {'Risk Contrib':>15} {'% of Risk':>12}")
print("-" * 70)
for ticker, weight, risk_c in zip(tickers, weights, marginal_contrib):
    print(f"{ticker:<10} {weight:>9.1%} {risk_c:>14.2%} {risk_c/marginal_contrib.sum():>11.1%}")
print("-" * 70)
print(f"{'TOTAL':<10} {weights.sum():>9.1%} {marginal_contrib.sum():>14.2%} {100:>11.0f}%")

print("\n" + "=" * 70)
print("Analysis complete!")

## 11. Practical Portfolio Analysis Example

Let's apply everything we've learned to a real-world portfolio analysis scenario.

In [ ]:
# Create comprehensive visualization dashboard
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Price and moving averages
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(df.index, df['Close'], label='Close Price', linewidth=2, color='black', alpha=0.7)
ax1.plot(df.index, df['SMA_20'], label='SMA 20', linewidth=1.5, color='blue', alpha=0.8)
ax1.plot(df.index, df['SMA_50'], label='SMA 50', linewidth=1.5, color='red', alpha=0.8)
ax1.plot(df.index, df['SMA_200'], label='SMA 200', linewidth=1.5, color='green', alpha=0.8)
ax1.fill_between(df.index, df['BB_Lower'], df['BB_Upper'], alpha=0.2, color='gray', label='Bollinger Bands')
ax1.set_title('Price Evolution with Moving Averages & Bollinger Bands', fontsize=14, fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Price ($)')
ax1.legend(loc='best', framealpha=0.9)
ax1.grid(True, alpha=0.3)

# 2. Volume analysis
ax2 = fig.add_subplot(gs[1, 0])
colors = ['green' if df['Price_Change_Pct'].iloc[i] >= 0 else 'red' for i in range(len(df))]
ax2.bar(df.index, df['Volume'] / 1e6, color=colors, alpha=0.6, width=0.8)
ax2.axhline(y=df['Volume'].mean() / 1e6, color='blue', linestyle='--', linewidth=2, label='Avg Volume')
ax2.set_title('Daily Volume (Millions)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Volume (M)')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# 3. Return distribution
ax3 = fig.add_subplot(gs[1, 1])
returns_clean = df['Return'].dropna()
ax3.hist(returns_clean, bins=50, density=True, alpha=0.7, color='skyblue', edgecolor='black')
# Fit normal distribution
from scipy import stats
mu, sigma = returns_clean.mean(), returns_clean.std()
x = np.linspace(returns_clean.min(), returns_clean.max(), 100)
ax3.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label=f'Normal(μ={mu:.4f}, σ={sigma:.4f})')
ax3.axvline(x=mu, color='green', linestyle='--', linewidth=2, label=f'Mean: {mu:.4f}')
ax3.set_title('Return Distribution', fontsize=12, fontweight='bold')
ax3.set_xlabel('Daily Return')
ax3.set_ylabel('Density')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Rolling volatility
ax4 = fig.add_subplot(gs[1, 2])
ax4.plot(df.index, df['Volatility_20'] * 100, color='purple', linewidth=2)
ax4.fill_between(df.index, 0, df['Volatility_20'] * 100, alpha=0.3, color='purple')
ax4.set_title('20-Day Rolling Volatility (Annualized)', fontsize=12, fontweight='bold')
ax4.set_xlabel('Date')
ax4.set_ylabel('Volatility (%)')
ax4.grid(True, alpha=0.3)

# 5. Cumulative returns
ax5 = fig.add_subplot(gs[2, 0])
cumulative_returns = (1 + df['Return']).cumprod()
ax5.plot(df.index, cumulative_returns, linewidth=2, color='darkgreen')
ax5.fill_between(df.index, 1, cumulative_returns, alpha=0.3, color='green')
ax5.axhline(y=1, color='black', linestyle='-', linewidth=1)
ax5.set_title('Cumulative Returns (Growth of $1)', fontsize=12, fontweight='bold')
ax5.set_xlabel('Date')
ax5.set_ylabel('Portfolio Value ($)')
ax5.grid(True, alpha=0.3)

# 6. Monthly returns heatmap
ax6 = fig.add_subplot(gs[2, 1])
monthly_ret = df.resample('M')['Return'].apply(lambda x: (1 + x).prod() - 1)
monthly_pivot = pd.DataFrame({
    'Month': monthly_ret.index.month,
    'Year': monthly_ret.index.year,
    'Return': monthly_ret.values * 100
}).pivot(index='Month', columns='Year', values='Return')
sns.heatmap(monthly_pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0, 
            cbar_kws={'label': 'Return (%)'}, ax=ax6)
ax6.set_title('Monthly Returns Heatmap (%)', fontsize=12, fontweight='bold')
ax6.set_ylabel('Month')
ax6.set_xlabel('Year')

# 7. Day of week analysis
ax7 = fig.add_subplot(gs[2, 2])
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
dow_returns = df.groupby('Day_of_Week')['Return'].mean() * 100
dow_returns.index = day_names
colors_dow = ['green' if x >= 0 else 'red' for x in dow_returns.values]
ax7.bar(dow_returns.index, dow_returns.values, color=colors_dow, alpha=0.7, edgecolor='black')
ax7.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax7.set_title('Average Return by Day of Week', fontsize=12, fontweight='bold')
ax7.set_xlabel('Day')
ax7.set_ylabel('Avg Return (%)')
ax7.grid(True, alpha=0.3, axis='y')

plt.suptitle('Comprehensive Financial Data Analysis Dashboard', 
             fontsize=16, fontweight='bold', y=0.995)
plt.show()

print("Dashboard created successfully!")